


## Table of Contents
- Part 1:  [Exploratory Data Analysis (EDA) - none NLP](#2)   
  - 1.1  [Import Files and Load Data](#3)
  - 1.2  [Basic Data Inspectations](#4)
  - 1.3  [Deeper EDA](#5)
    - 1.3.1 [Dependent Variable Distribution](#6)
    - 1.3.2 [Null/Missing Values and feature analysis](#7)
- Part 2:  [Exploratory Data Analysis (EDA) - NLP](#8)
  - 2.1:  [Tweets, Word Clouds and Unique Words](#9)
  - 2.2: [NLP Meta Features](#10)
  - 2.3: [N Grams](#11)
- Part 3: [Data Preprocessing](#12)
- Part 4: [Model Building and Training](#15)
  - 4.1 [Baseline Model](#16)
  - 4.2 [Model Tuning 1 - Recurrent Model LSTM](#17)
  - 4.3 [Model Tuning 2 - Add GloVe Embedding](#18)
  - 4.4 [Model Tuning 3 - Recurrent GRU Model](#19)
- Part 5: [Submit Testing Results](#20)
- Part 6: [Conclusion](#21)
- [References](#22)
<br>


##### <a id="2"></a>
<div style="text-align: center; background-color: #527506; font-size:80%; font-family: Comic Sans MS; padding: 5px;border-radius:10px 10px; color: white;"> 
    <h1>Part 1: Exploratory Data Analysis (EDA) 
🍂</h1>

</div>

##### <a id="3"></a>

<div style="text-align: center; background-color: #c3c20e; font-size:70%; font-family: Comic Sans MS; padding: 2px;border-radius:10px;">
    <h2>Part 1.1: Import Files and Load Data</h2>
</div>

In [ ]:
# Import files
import os
import numpy as np 
import pandas as pd 
import itertools
from scipy.stats import norm, skew, mode
import re
import gc
import string
import warnings 
import operator

import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.initializers import Constant
from tensorflow.keras.callbacks import EarlyStopping
from keras.metrics import Precision, Recall

from sklearn.feature_extraction.text import  CountVectorizer
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import train_test_split

In [ ]:
# 모든 무작위 시드 고정
SEED_VALUE = 998 # 원하는 시드 값
os.environ['PYTHONHASHSEED'] = str(SEED_VALUE)
import random
random.seed(SEED_VALUE)
import numpy as np
np.random.seed(SEED_VALUE)
tf.random.set_seed(SEED_VALUE)

# TensorFlow의 결정론적 동작 활성화 (GPU에서도 재현성 보장 시도)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1' # CuDNN 관련 (LSTM, GRU 등)
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.threading.set_intra_op_parallelism_threads(1)

# GPU 메모리 증가 허용 (ResourceExhaustedError 방지를 위함)
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

In [ ]:
# Load data
root_dir = '/kaggle/input/nlp-getting-started'

train=pd.read_csv(os.path.join(root_dir, 'train.csv'))
train2=pd.read_csv(os.path.join('/kaggle/input/duplicated-data/given_data_augmented.csv'))



test=pd.read_csv(os.path.join(root_dir, 'test.csv'))
sample_submission=pd.read_csv(os.path.join(root_dir, 'sample_submission.csv'))


##### <a id="4"></a>

<div style="text-align: center; background-color: #c3c20e; font-size:70%; font-family: Comic Sans MS; padding: 2px;border-radius:10px;">
    <h2>Part 1.2: Basic Data Inspections</h2>
</div>

In [ ]:
train.head()

In [ ]:
#train = pd.concat([train, train2], axis=0)

In [ ]:
#Check for duplicate values
print(len(train))
train= train.drop_duplicates(keep='first')
print(len(train))
dataset_size=len(train)

In [ ]:
train[train.duplicated()]

In [ ]:
print('************************************training data******************************************')
print(train.info())
print('************************************testing data******************************************')
print(test.info())

In [ ]:
train.describe(include='all')


### Basic EDA observations ⛈

- Training data has **7613 rows and 5 columns**. Testing data has **3263 rows and 4 columns**. The difference is training data has a 'target' column, which indicates if the twit is disaster or non-disaster.
- There are **61 'keyword' and 2472 'location' null values** in training data. There are **26 'keyword' and 1150 'location' null values** in training data.


##### <a id="5"></a>

<div style="text-align: center; background-color: #c3c20e; font-size:70%; font-family: Comic Sans MS; padding: 2px;border-radius:10px;">
    <h2>Part 1.3: Deeper EDA - non NLP</h2>
</div>

##### <a id="6"></a>

<div style="text-align: center; background-color: #70ced0; font-size:70%; font-family: Comic Sans MS; padding: 2px;border-radius:10px;">
    <h2>Part 1.3.1: Dependent Variable Distribution</h2>
</div>

In [ ]:
# check the balance of dependent variable
plt.figure(figsize=(4, 3))
train.target.value_counts().plot(kind='bar',color='skyblue')
plt.title(f'Positive 1 (Disaster Tweets) Percentage: {round(train.target.value_counts()[1]/train.shape[0]*100,0)}%')

Seems in training set, the data is a bit imbalanced (but not terrible) with around **43% positive (disaster tweets) samples**. Since the metric for this competition is F1 score ($ \frac{2}{\frac{1}{\text{precision}} + \frac{1}{\text{recall}}} $), which is around the average of precision and recall, a little dependent variable imbalance is probably fine.

In [ ]:
# Create 'message_length' field in copy of train data
def plot_message_length(df):
        df_copy = df.copy()
        df_copy['message_length'] = df_copy['text'].apply(lambda x: len(x.split(" ")))
        print("None Disaster Tweet Message Length skewness: %.2f and kurtosis: %.2f" % 
              (df_copy[df_copy['target'] == 0]['message_length'].skew(), 
               df_copy[df_copy['target'] == 0]['message_length'].kurt()))

        print("Disaster Tweet Message Length skewness: %.2f and kurtosis: %.2f" % 
              (df_copy[df_copy['target'] == 1]['message_length'].skew(), 
               df_copy[df_copy['target'] == 1]['message_length'].kurt()))

        #Check distribution 
        plt.figure(figsize=(30,3))
        with pd.option_context('mode.use_inf_as_na', True):
            plt.subplot(1, 2, 1)
            sns.histplot(df_copy[df_copy['target'] == 0]['message_length'], kde=True, bins=20, color='y')
            plt.title("Non-Disaster tweets")

            plt.subplot(1, 2, 2)
            sns.histplot(df_copy[df_copy['target'] == 1]['message_length'], kde=True, bins=20, color='goldenrod')
            plt.title("Disaster tweets")
  
        plt.figure(figsize=(12, 2))
        sns.boxplot(data=df_copy, y='target', x='message_length', palette={0: 'y', 1: 'goldenrod'}, orient='h')
        plt.xlabel('Message Length')
        plt.ylabel('Disaster vs. Non-Disaster')
        
        plt.tight_layout()
        plt.show()
        
plot_message_length(train)

- Both disaster and on-disaster tweets are short texts and the **majority of them have 0 to 30 words and centered around 15**. They have nearly symmetric distributions.
- The **disaster tweets** distribution has **heavier tails and a sharper peak** compared to the normal distribution. But both of them are almost symetric and the peak's sharpness are closer to a normal distribution (|skewness|<<1 and |kurtosis|<<3)

##### <a id="7"></a>

<div style="text-align: center; background-color: #70ced0; font-size:70%; font-family: Comic Sans MS; padding: 2px;border-radius:10px;">
    <h2>Part 1.3.2: Null/Missing Values and feature analysis</h2>
</div>

In [ ]:
# check null values
print('************************************training data******************************************')
print('null values:\n',train.isnull().agg(['sum','mean'], axis=0))
print('Duplicated rows: ',train.duplicated().sum(axis=0))
print('Duplicated IDs: ',train.id.duplicated().sum(axis=0))
print('************************************testing data******************************************')
print('null values:\n',test.isnull().agg(['sum','mean'], axis=0))
print('Duplicates rows: ',train.duplicated().sum(axis=0))
print('Duplicated IDs: ',test.id.duplicated().sum(axis=0))
train_pos=train[train['target']==1]
train_neg=train[train['target']==0]
print('************************************training data - non-disaster******************************************')
print('null values:\n',train_pos.isnull().agg(['sum','mean'], axis=0))
print('************************************training data - disaster ******************************************')
print('null values:\n',train_neg.isnull().agg(['sum','mean'], axis=0))


- Overall, we have around 0.8% keyword and 33.3% location missing. 
- Disaster tweets have a bit more keyword missing 1.3% (vs. 0.4% for non-disaster)

### 'location'feature deeper look

In [ ]:
# check if 'location' missing values are positive or negative in 'target' value
train.groupby('target')['location'].apply(lambda x: x.isnull().sum()).plot.bar(figsize=(4,3),title="Location missing values in non-disater vs. disaster tweets",color='gray')

In [ ]:
# 'Location' distribution
print(f'Number of unique locations: {len(train.location.unique())}')
plt.figure(figsize=(30,3))
train.location.value_counts().head(60).plot(kind='bar',color='gray',fontsize=20)
plt.show()

Seems **'location' field** is **a bit messy**. Some tweets has state and some don't, some typos, and different 'location' represent the same thing (for example, 'NYC' vs. "New York"). I am not sure this field would contribute to predicting the positive or negative disaster tweets or just add noise. We will probably need to **exclude** this.

### 'keyword'feature deeper look

In [ ]:
# Unique keywords
print('unique keywords in training:', len(train['keyword'].unique()))
print('unique keywords in testing:', len(test['keyword'].unique()))
print('unique keywords in testing not in training:', len( set(test['keyword'].unique()) - set(train['keyword'].unique())))
print('unique keywords in trainingnot in testing :', len( set(train['keyword'].unique()) - set(test['keyword'].unique())))

- **'keyword' feature only has one word per record**. 
- There are **222 unique ones** in each **training and testing** data and they have the **same unique keywords**. 

 Those probably means the keyword converys very important information. Let's take a closer look at keyword distributions in disaster vs. non-disaster tweets.

In [ ]:
# Plot keywords by disaster (1) vs. non-disaster (0) tweets

train_no_disaster = train[train['target']==0]
train_disaster = train[train['target']==1]

plt.figure(figsize=[25, 12])
plt.subplot(1,3,1)
sns.countplot(y='keyword', 
              data=train,
              order=train['keyword'].value_counts()[:30].index,
              hue='target',
              palette=['y', 'darkorange']).set(title='Top 30 Keywords Distributions', 
                                                      ylabel='Keywords')
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['top'].set_visible(False)
plt.legend()
plt.subplot(1,3,2)
sns.countplot(y='keyword',order= train_no_disaster['keyword'].value_counts().index[:30],data=train_no_disaster, palette=['y']).set(title='Top 30 Keywords Distributions - NonDisaster Tweets',ylabel='')
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['top'].set_visible(False)
plt.subplot(1,3,3)
sns.countplot(y='keyword', order = train_disaster['keyword'].value_counts().index[:30], data=train_disaster, palette=['darkorange']).set(title='Top 30 Keywords Distributions - Disaster Tweets',ylabel='')
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['top'].set_visible(False)
plt.show();

- It does seem for most keywords, the distributions are **quite different for disaster vs. non-disaster tweets**. This confirms that this field could indeed really helpful for our prediction!We might append this to end of main 'text' field for prediction.



### Deeper EDA (non- NLP specific observations) summary ⛈
    
- Target variable is a bit imbalanced (but not terrible) with around **43% positive samples**. It's probably fine since the competition is going to use F1 score, which is balanced between precision and recall. 
- The majority of positive and negative target groups have **0 to 30 words**. 
- The **positive target group**'s distribution has **heavier tails and a sharper peak** compared to the normal distribution, while the negative target group's distribution has lighter tails and a flatter peak.
- **Disaster tweets have a bit more keyword missing 1.3%** (vs. 0.4% for non-disaster)
- **'location'** field is a bit **messy and might not useful**.
- **'key word'** field has one keyword per record. There are **the same 222 unique keywords in both training and testing data**. Those keywords **convery important information** about tweet contents. 
-  However, I've **tried to append 'keyword'** to the end of the text to improve model prediction performance. While it doens't hurt the performance too much, **it doesn't help**. I will leave them out (and for simplicity I've deleted the part I used them). :(

##### <a id="8"></a>
<div style="text-align: center; background-color: #527506; font-size:80%; font-family: Comic Sans MS; padding: 5px;border-radius:10px 10px; color: white;"> 
    <h1>Part 2: Exploratory Data Analysis (EDA) - NLP
☃️</h1>

</div>

##### <a id="9"></a>

<div style="text-align: center; background-color: #c3c20e; font-size:70%; font-family: Comic Sans MS; padding: 2px;border-radius:10px;">
    <h2>Part 2.1:  Tweets, Word Clouds and Unique Words</h2>
</div>

In [ ]:
# Look at tweets
print('********************************Disaster tweets********************************')
print(train[train['target']== 1]['text'].iloc[0],'\n')
print(train[train['target']== 1]['text'].iloc[1],'\n')
print(train[train['target']== 1]['text'].iloc[2],'\n')

print('********************************Non-Disaster tweets********************************')
print(train[train['target']== 0]['text'].iloc[0],'\n')
print(train[train['target']== 0]['text'].iloc[1],'\n')
print(train[train['target']== 0]['text'].iloc[2],'\n')


In [ ]:
# Word Cloud 

def generate_wordcloud(df, start, end, colormap='viridis',target='', word='Words'):
    frequency = df.str.split().explode().value_counts()
    words = frequency.index[start:end].tolist()
    counts = frequency.values[start:end].tolist()
    wordcloud_dict = {words[i]: counts[i] for i in range(len(words))}
    wordcloud = WordCloud(width=800, height=400, background_color='white',colormap=colormap).generate_from_frequencies(wordcloud_dict)
    
    plt.figure(figsize=(12, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.title(f'{target} Tweets Most Frequent {word}: Top {start+1} to {end}')
    plt.axis('off')
    plt.show()

generate_wordcloud(train['text'], 0, 500, colormap= 'gray')

Many words are stop words and they aren't meaningful. However, they might help to determine the meaning of sentences in sequences.

In [ ]:
# Unqiue words
def unique_words_train_test(col, n=600):
        # First let's check number of unique words in training and testing
        unique_words_train = len(train[col].str.split(" ").explode().unique())
        unique_words_test = len(test[col].str.split(" ").explode().unique())

        # Find words in test not in train
        unique_words_train_list = set(train[col].str.split().explode().unique())
        unique_words_test_list = set(test[col].str.split().explode().unique())
        words_in_test_not_train = list(unique_words_test_list.difference(unique_words_train_list))
        words_in_train_not_test = list(unique_words_train_list.difference(unique_words_test_list))
        words_in_either_train_and_test = list(unique_words_train_list.union(unique_words_test_list))
        words_in_both_train_and_test = list(unique_words_train_list.intersection(unique_words_test_list))
        
        # Unique words in training and testing 
        print("Words in training:", unique_words_train)
        print("Words in testing:", unique_words_test)
        print("Words in testing not in training:", len(words_in_test_not_train))
        print("Words in either testing and training:", len(words_in_either_train_and_test))
        print("Words in both testing and training:", len(words_in_both_train_and_test))

        # plot
        plt.figure(figsize=(15,3))
        plt.bar(['Train', 'Test','In Test Not In Train','In Train Not In Test','In Either','In Both'], [unique_words_train, unique_words_test, len(words_in_test_not_train), len(words_in_train_not_test), len(words_in_either_train_and_test), len(words_in_both_train_and_test)], 
                 color=['skyblue', 'mediumpurple','red','gray','green','blue'])
        plt.title('Count of Unique Words in Train and Test Data')
        plt.ylabel('Count of Unique Words')
        plt.show()
        
        print("Some sample words in testing not in training:\n", words_in_test_not_train[:100])
unique_words_train_test('text')

    
### Unique Words observations ⛈
    
Although the count of unique words in training and testing data are similar. There are **so many words in testing that aren't in training**. This isn't good because the model wouldn't recognize those words in testing time if we use normal embedding. Let's take a look at what those discrepancy words are. 

I can see a few reasons/issues:
- Things shouldn't be looked as unique words. 
  - **Hyperlinks**: usually starts with 'http:','https:',
  - **Numbers**: "01:02:42", "1945", "40's" for example.
  - **Mentions**: usually start with '@' 
- Things combine with 'word' were treated as a different word:
  - **Punctuations**: 'bags:','Facebook.','now.', 'tonight??', 'meals*', './.....hmm', '\x89Û÷crying' for example. Stemming won't be able to fix this. 
  - **Hashtags**:  '#LA', '#Bombeffects','#German'
- Words in different forms:
  - Words in **different forms**:,'saying','counters', 'snowy' for example.
  - Words in **different higher and lower cases**: 'DYING', 'Amase', 'TALK' for example
  - Typos and repeated single characters:'Nooooooooooooooo','Yesssss', for example
- Special characters:
  - 'her\x89Û_', '#1\x89ÛÒDon\x89Ûªt' for example

  
Okay, overall, it seems we need to do some corpus cleaning!! Before that, let's look at create and look at some NLP "Meta Features".
  

##### <a id="10"></a>

<div style="text-align: center; background-color: #c3c20e; font-size:70%; font-family: Comic Sans MS; padding: 2px;border-radius:10px;">
    <h2>Part 2.2:  NLP Meta Features </h2>
</div>

<div style="background-color: #effdee; padding: 10px;">
<span style="font-size: larger;">
    
#### Meta Features

In NLP, "Meta-features" are additional features or characteristics derived from the data that are used to augment the original dataset for analytical and modeling purposes. Inspired by this [notebook](https://www.kaggle.com/code/gunesevitan/nlp-with-disaster-tweets-eda-cleaning-and-bert), **disaster tweets tend to have more tweets that are writen in a more formal way** by news agencies.
    
Let's create some 'Meta Features" for disaster and non-disaster tweets. Also, we'll look at their distributions in training vs. tesings set because the distribution discrepancy between training and testing can affect testing predictions:
- **word_count**: number of words in text
- **unique_word_count**: number of unique words in text
- **stop_word_count**: number of stop words in text
- **url_count**: number of urls in text
- **mean_word_length**: average character count in words
- **char_count**:number of characters in text
- **punctuation_count**: number of punctuations in text
- **hashtag_count**: number of hashtags (#) in text
- **mention_count**: number of mentions (@) in text

In [ ]:
# Source code https://www.kaggle.com/code/gunesevitan/nlp-with-disaster-tweets-eda-cleaning-and-bert
# word_count
train['word_count'] = train['text'].apply(lambda x: len(str(x).split()))
test['word_count'] = test['text'].apply(lambda x: len(str(x).split()))

# unique_word_count
train['unique_word_count'] = train['text'].apply(lambda x: len(set(str(x).split())))
test['unique_word_count'] = test['text'].apply(lambda x: len(set(str(x).split())))

# stop_word_count, I will add a few more I saw in data
stop_words = nltk.corpus.stopwords.words('english')
train['stop_word_count'] = train['text'].apply(lambda x: len([w for w in str(x).lower().split() if w in stop_words]))
test['stop_word_count'] = test['text'].apply(lambda x: len([w for w in str(x).lower().split() if w in stop_words]))

# url_count
train['url_count'] = train['text'].apply(lambda x: len([w for w in str(x).lower().split() if 'http' in w or 'https' in w or 'www' in w]))
test['url_count'] = test['text'].apply(lambda x: len([w for w in str(x).lower().split() if 'http' in w or 'https' in w or 'www' in w]))

# mean_word_length
train['mean_word_length'] = train['text'].apply(lambda x: np.mean([len(w) for w in str(x).split()]))
test['mean_word_length'] = test['text'].apply(lambda x: np.mean([len(w) for w in str(x).split()]))

# char_count
train['char_count'] = train['text'].apply(lambda x: len(str(x)))
test['char_count'] = test['text'].apply(lambda x: len(str(x)))

# punctuation_count
train['punctuation_count'] = train['text'].apply(lambda x: len([c for c in str(x) if c in string.punctuation]))
test['punctuation_count'] = test['text'].apply(lambda x: len([c for c in str(x) if c in string.punctuation]))

# hashtag_count
train['hashtag_count'] = train['text'].apply(lambda x: len([c for c in str(x) if c == '#']))
test['hashtag_count'] = test['text'].apply(lambda x: len([c for c in str(x) if c == '#']))

# mention_count
train['mention_count'] = train['text'].apply(lambda x: len([c for c in str(x) if c == '@']))
test['mention_count'] = test['text'].apply(lambda x: len([c for c in str(x) if c == '@']))

In [ ]:
# Source code https://www.kaggle.com/code/gunesevitan/nlp-with-disaster-tweets-eda-cleaning-and-bert

# histograms for Meta-Features

warnings.filterwarnings("ignore", message="use_inf_as_na option is deprecated", category=FutureWarning) # Suppress FutureWarning


METAFEATURES = ['word_count', 'unique_word_count', 'stop_word_count', 'url_count', 'mean_word_length',
                'char_count', 'punctuation_count', 'hashtag_count', 'mention_count']
DISASTER_TWEETS = train['target'] == 1

fig, axes = plt.subplots(ncols=2, nrows=len(METAFEATURES), figsize=(20, 40), dpi=100)

for i, feature in enumerate(METAFEATURES):
    sns.histplot(train.loc[~DISASTER_TWEETS,feature], label='Non-Disaster', ax=axes[i,0], color='y', kde=True)
    sns.histplot(train.loc[DISASTER_TWEETS,feature], label='Disaster', ax=axes[i,0], color='darkorange', kde=True)

    sns.histplot(train[feature], label='Training', ax=axes[i,1], color='skyblue', kde=True)
    sns.histplot(test[feature], label='Test', ax=axes[i,1], color='thistle', kde=True)
    
    for j in range(2):
        axes[i,j].set_xlabel('')
        axes[i,j].tick_params(axis='x', labelsize=12)
        axes[i,j].tick_params(axis='y', labelsize=12)
        axes[i,j].set_title(f'{feature} Distribution in Training Set', fontsize=13)
        axes[i,j].set_title(f'{feature} Distribution in Training & Test Set ', fontsize=13)
        axes[i,j].legend()
        
plt.show()

### Meta Features observations ⛈
    
From the plots above, alhtough most of the distributions look very similar in disaster vs. non-disaster tweets, we **saw a trend that disaster tweets do tend to have more formal ones**:
- **Non-Disaster tweets have more meanion tags @**
- **Disaster tweets use slightly longer words** ('mean_word_length' is bigger in disaster tweets).
- **Disaster tweets have a bit less low count punctuation (<5) and low count character (<60)** tweets

Also:
- **disaster tweets have more url links**  (more >=1 'url_count' in disaster tweets)
- For some reason, training data seems to have more high character count (around 140) tweets.  

##### <a id="11"></a>

<div style="text-align: center; background-color: #c3c20e; font-size:70%; font-family: Comic Sans MS; padding: 2px;border-radius:10px;">
    <h2>Part 2.3:  N Grams </h2>
</div>

In [ ]:
def ngram_freqplot(df, color='skyblue', target=" "):
    fig, axs = plt.subplots(1, 3, figsize=(20,10))  
    for i in range(1, 4):
        # Use CountVectorizer to get ngram
        vectorizer = CountVectorizer(ngram_range=(i, i), stop_words='english')
        ngram_train = vectorizer.fit_transform(df['text'])

        # Calculate the frequency of each n-gram and store in dictionary 
        ngram_freq = ngram_train.sum(axis=0)
        feature_names = vectorizer.get_feature_names_out()
        ngram_dict = {feature_names[j]: ngram_freq[0, j] for j in range(len(feature_names))}
        
        # Filter out n-grams containing URLs
        ngram_dict = {ngram: freq for ngram, freq in ngram_dict.items() if not re.match(r'(http|https)', ngram)}
        
        sorted_ngrams = sorted(ngram_dict.items(), key=lambda x: x[1], reverse=True)

        # top n-grams 
        top_n = 30  # Adjust the number of top n-grams you want to plot
        top_ngrams = [item[0] for item in sorted_ngrams[:top_n]]
        top_frequencies = [item[1] for item in sorted_ngrams[:top_n]]
        print(top_ngrams)

        # bar plot
        axs[i-1].barh(top_ngrams, top_frequencies, color=color)
        axs[i-1].set_title(f'{target} tweet top 30 {i}-gram ')
        axs[i-1].invert_yaxis()  # want highest frequency at the top
        axs[i-1].tick_params(axis='x') 

        # font size for the words 
        for tick in axs[i-1].get_yticklabels():
            tick.set_fontsize(15)
    plt.tight_layout()

In [ ]:
# Disaster tweets
ngram_freqplot(train[DISASTER_TWEETS], color='goldenrod', target="Disaster")

In [ ]:
# non-disaster tweets
ngram_freqplot(train[~DISASTER_TWEETS], color='y', target ='non-disaster')

- Some related disaster vs. non-disaster unigram and bigram. For example, 'disaster', 'suiside', 'oil spill' are frequent in disaster tweets. While in non-disaster tweets, the frequent words are more light-hearted. I think those would help with predictions. 
- However, many frequent words in disaster and non-disaster tweets don't seem to be helpful. For example, 'like' are frequent in both kinds of tweets.


# <a id="12"></a>
<div style="text-align: center; background-color: #527506; font-size:80%; font-family: Comic Sans MS; padding: 5px;border-radius:10px 10px; color: white;"> 
    <h1>Part 3: Data Preprocessing☔️ </h1>

</div>

In [ ]:

import re
import string

# 1. 축약어 확장 맵 (예시)
# 영어 트윗 데이터셋에 맞게 더 많은 축약어를 추가할 수 있습니다.
CONTRACTION_MAP = {
    "ain't": "is not", "aren't": "are not", "can't": "cannot", "can't've": "cannot have",
    "'cause": "because", "could've": "could have", "couldn't": "could not",
    "didn't": "did not", "doesn't": "does not", "don't": "do not", "hadn't": "had not",
    "hasn't": "has not", "haven't": "have not", "he'd": "he would", "he'll": "he will",
    "he's": "he is", "how'd": "how did", "how'll": "how will", "how's": "how is",
    "i'd": "i would", "i'd've": "i would have", "i'll": "i will", "i'll've": "i will have",
    "i'm": "i am", "i've": "i have", "isn't": "is not", "it'd": "it would",
    "it'll": "it will", "it's": "it is", "let's": "let us", "ma'am": "madam",
    "mightn't": "might not", "must've": "must have", "n't": " not", "needn't": "need not",
    "she'd": "she would", "she'll": "she will", "she's": "she is", "should've": "should have",
    "shouldn't": "should not", "that'd": "that would", "that's": "that is",
    "there'd": "there would", "there's": "there is", "they'd": "they would",
    "they'll": "they will", "they're": "they are", "they've": "they have",
    "we'd": "we would", "we're": "we are", "we've": "we have", "weren't": "were not",
    "what'll": "what will", "what're": "what are", "what's": "what is",
    "where's": "where is", "who'd": "who would", "who'll": "who will", "who's": "who is",
    "won't": "will not", "wouldn't": "would not", "you'd": "you would", "you'll": "you will",
    "you're": "you are"
}

def clean_and_normalize_text(text):
    """
    트윗 텍스트에 대해 노이즈 처리와 정규화 작업을 수행하는 함수.
    """
    text = str(text)  # 입력이 문자열인지 확인

    # --- 1. 노이즈 처리 강화 ---

    # 1.1 URL 대체 및 제거 (URL을 <URL> 토큰으로 대체하여 모델이 그 존재를 특징으로 학습하게 함)
    text = re.sub(r'https?://\S+|www\.\S+', '<URL>', text) 

    # 1.2 사용자 멘션 대체
    text = re.sub(r'@\w+', '<MENTION>', text) 
    
    # 1.3 HTML 태그 및 엔티티 제거 (예: &amp; &lt; &gt;)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'&[a-z]+;', '', text) 

    # 1.4 해시태그 심볼 제거 (단어는 유지)
    text = re.sub(r'#', '', text) 
    
    # 1.5 의미 없는 특수 문자, 이스케이프 시퀀스 제거 (예: \x89Û_)
    text = re.sub(r'\\x[0-9A-Fa-f]{2}|[^a-zA-Z0-9\s.,?!<>()\'\"]', ' ', text)
    
    # --- 2. 텍스트 정규화 ---

    # 2.1 대소문자 통일 (모두 소문자로)
    text = text.lower()
    
    # 2.2 축약어 확장 (Contraction Expansion)
    # 맵에 정의된 축약어를 확장
    words = text.split()
    expanded_words = []
    for word in words:
        if word in CONTRACTION_MAP:
            expanded_words.extend(CONTRACTION_MAP[word].split())
        else:
            expanded_words.append(word)
    text = ' '.join(expanded_words)
    
    # 2.3 반복 문자 줄이기 (예: nooooooo -> nooo)
    # 3개 이상의 반복 문자를 2개로 줄임. (예: 'wiiild' -> 'wiild')
    text = re.sub(r'([a-z])\1{2,}', r'\1\1', text) 
    
    # 2.4 불필요한 공백 제거 (다중 공백을 단일 공백으로 치환하고 앞뒤 공백 제거)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# 예시 적용
tweet_example = "OMG! Tsunami is coming! Check out this link: http://bit.ly/1234. @john_doe you'll never believe this!!! Noooooooo. I'm afraid &amp; I can't do anything."
print("원본 트윗: ", tweet_example)
cleaned_tweet = clean_and_normalize_text(tweet_example)
print("정규화된 트윗: ", cleaned_tweet)

In [ ]:

# Text preprocessing
train['clean_text'] = train['text'].apply(clean_and_normalize_text)
test['clean_text'] = test['text'].apply(clean_and_normalize_text)


In [ ]:
# Now let's take a look again the unique words in training and testing data
unique_words_train_test('clean_text',n=100)

###### <a id="15"></a>
<div style="text-align: center; background-color: #527506; font-size:80%; font-family: Comic Sans MS; padding: 5px;border-radius:10px 10px; color: white;"> 
    <h1>Part 4: Model Building and Training 🚑 </h1>

</div>

In [ ]:
# Train_test split
TEST_SIZE=0.2
y = train['target']
X_train, X_val, y_train, y_val = train_test_split(train, 
                                                      y, 
                                                      test_size=TEST_SIZE, 
                                                      random_state=100) 

OOV_TOKEN ="<OOV>"

# Initialize the Tokenizer class
tokenizer = Tokenizer(num_words= 21884 , split=' ', oov_token=OOV_TOKEN)

# Tokenizer using Tensorflow Keras
VOCAB_SIZE = len(tokenizer.word_index) + 1  #unique words in training


# Generate the word index dictionary for the training sentences
tokenizer.fit_on_texts(X_train['clean_text']) 
word_index = tokenizer.word_index #for later glove embedding

# Generate and pad sequences
sequences_train = tokenizer.texts_to_sequences(X_train['clean_text'].values) #.values convert pd series to numpy array
padded_train = pad_sequences(sequences_train)
max_length = max(len(seq) for seq in padded_train)# Find maximum sequence length

sequences_val = tokenizer.texts_to_sequences(X_val['clean_text'].values)
padded_val = pad_sequences(sequences_val, maxlen=max_length, truncating='post')

sequences_test = tokenizer.texts_to_sequences(test['clean_text'].values)
padded_test = pad_sequences(sequences_test,maxlen=max_length, truncating='post')

print("Maximum sequence length:", max_length)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

hashtag_train = scaler.fit_transform(X_train['hashtag_count'].values.reshape(-1, 1))
hashtag_val = scaler.transform(X_val['hashtag_count'].values.reshape(-1, 1))
hashtag_test = test['hashtag_count'].values.reshape(-1, 1)

# 추가: 구두점 수 데이터
punctuation_train = scaler.fit_transform(X_train['punctuation_count'].values.reshape(-1, 1))
punctuation_val = scaler.transform(X_val['punctuation_count'].values.reshape(-1, 1))
punctuation_test = test['punctuation_count'].values.reshape(-1, 1)

# 추가: URL 수 데이터
url_train = scaler.fit_transform(X_train['url_count'].values.reshape(-1, 1))
url_val = scaler.transform(X_val['url_count'].values.reshape(-1, 1))
url_test = test['url_count'].values.reshape(-1, 1)

print("해시태그 트레인 데이터 형태:", hashtag_train.shape)
print("구두점 트레인 데이터 형태:", punctuation_train.shape)
print("URL 트레인 데이터 형태:", url_train.shape)

In [ ]:
# Plot results function
def plot_results(history, model_name="", early_stopping_callback=None):
    metrics = ['accuracy', 'loss', 'precision', 'recall']
    titles = [f'{model_name} Accuracy', f'{model_name} Loss', f'{model_name} Precision', f'{model_name} Recall', f'{model_name} F1 Score']
        
    # Calculate F1 score for training and validation data
    train_precision = history.history['precision']
    train_recall = history.history['recall']
    train_f1_scores = [2 * (train_precision[i] * train_recall[i]) / (train_precision[i] + train_recall[i] + 1e-8)
                   for i in range(len(train_precision))]  # Avoid division by zero
    
    val_precision = history.history['val_precision']
    val_recall = history.history['val_recall']
    val_f1_scores = [2 * (val_precision[i] * val_recall[i]) / (val_precision[i] + val_recall[i] + 1e-8)
                     for i in range(len(val_precision))]  # Avoid division by zero
    
    fig, axs = plt.subplots(3, 2, figsize=(20, 15))

    for i, metric in enumerate(metrics):
        row = i // 2
        col = i % 2
        axs[row, col].plot(history.history[metric])
        axs[row, col].plot(history.history[f'val_{metric}'])
        axs[row, col].set_title(titles[i], fontsize=20)  
        axs[row, col].tick_params(axis='both', which='major', labelsize=20)  
        axs[row, col].legend(['Train', 'Validate'], loc='upper left', fontsize=20)

    # Plot F1 score 
    axs[-1, 0].plot(train_f1_scores)
    axs[-1, 0].plot(val_f1_scores)
    axs[-1, 0].set_title(titles[-1], fontsize=20)  
    axs[-1, 0].tick_params(axis='both', which='major', labelsize=20)  
    axs[-1, 0].legend(['Train', 'Validate'], loc='upper left', fontsize=20)
 
    if early_stopping_callback is not None:
        # return best accuracy and corresponding metrics if early stopping
        val_accuracy = history.history['val_accuracy']
        best_index = np.argmax(val_accuracy)
        best_accuracy = val_accuracy[best_index]
        best_precision = val_precision[best_index]
        best_recall = val_recall[best_index]
        best_f1_score = val_f1_scores[best_index]
        return best_accuracy, best_precision, best_recall, best_f1_score
    else:
        # no early stopping callback
        return history.history['val_accuracy'][-1],  val_precision[-1],val_recall[-1],  val_f1_scores[-1]

    plt.tight_layout()
    plt.show()

In [ ]:
# Confusion matrix
def plot_cfm(model, X, y_actual, model_name=""):
        # predictions
        pred_prob = model.predict(X)
        pred_prob = np.transpose(pred_prob)[0]
        y_pred = list(map(lambda x: 0 if x < 0.5 else 1, pred_prob))

        #confusion Matrix
        cfm= confusion_matrix(y_val, y_pred)
        plt.figure(figsize=(1, 1))
        display = ConfusionMatrixDisplay(confusion_matrix=cfm) 
        display.plot(cmap='Greens')
        plt.title(f'{model_name} Confusion Matrix')
        plt.xlabel('Predicted Label')
        plt.ylabel('True Label')
        plt.gcf().set_size_inches(4, 3) 
        plt.show()
        return y_pred

# <a id="18"></a>
<div style="text-align: center; background-color: #c3c20e; font-size:70%; font-family: Comic Sans MS; padding: 2px;border-radius:10px;">
    <h2>Part 4.3:  Model Tuning 3 - Add GloVe Embedding to LSTM </h2>
</div>

In [ ]:
# load glove embeddings dictionary
glove_embeddings = np.load('/kaggle/input/pickled-glove840b300d-for-10sec-loading/glove.840B.300d.pkl', allow_pickle=True)

In [ ]:
# Source code: https://www.kaggle.com/code/gunesevitan/nlp-with-disaster-tweets-eda-cleaning-and-bert/notebook#4.-Embeddings-and-Text-Cleaning
# check word embedding vocabulary coverage

# Build vocabulary for tweets
def build_vocab(X):
    texts = X.apply(lambda s: s.split()).values      
    vocab = {}
    
    for text in texts: 
        for word in text: 
                vocab[word]=  vocab.get(word,0) + 1   
    print(len(vocab))
    return vocab

def check_embeddings_coverage(X, embeddings):
    vocab = build_vocab(X)    
    covered = {}
    oov = {}    
    n_covered = 0
    n_oov = 0
    
    for word in vocab:
        try:
            covered[word] = embeddings[word]
            n_covered += vocab[word]
        except:
            oov[word] = vocab[word]
            n_oov += vocab[word]
            
    vocab_coverage = len(covered) / len(vocab)
    text_coverage = (n_covered / (n_covered + n_oov))
    
    sorted_oov = sorted(oov.items(), key=operator.itemgetter(1))[::-1]
    return sorted_oov, vocab_coverage, text_coverage

train_glove_oov, train_glove_vocab_coverage, train_glove_text_coverage = check_embeddings_coverage(train['text'], glove_embeddings)
test_glove_oov, test_glove_vocab_coverage, test_glove_text_coverage = check_embeddings_coverage(test['text'], glove_embeddings)
print('GloVe Embeddings cover {:.2%} of vocabulary and {:.2%} of text in Training Set Before text processing'.format(train_glove_vocab_coverage, train_glove_text_coverage))
print('GloVe Embeddings cover {:.2%} of vocabulary and {:.2%} of text in Test Set Before text processing'.format(test_glove_vocab_coverage, test_glove_text_coverage))


train_glove_oov, train_glove_vocab_coverage, train_glove_text_coverage = check_embeddings_coverage(train['clean_text'], glove_embeddings)
test_glove_oov, test_glove_vocab_coverage, test_glove_text_coverage = check_embeddings_coverage(test['clean_text'], glove_embeddings)
print('GloVe Embeddings cover {:.2%} of vocabulary and {:.2%} of text in Training Set'.format(train_glove_vocab_coverage, train_glove_text_coverage))
print('GloVe Embeddings cover {:.2%} of vocabulary and {:.2%} of text in Test Set'.format(test_glove_vocab_coverage, test_glove_text_coverage))


In [ ]:
# Embedding matrix for GloVe word vectors

# get vacab size from embedding
VOCAB_SIZE = len(glove_embeddings)
EMBEDDED_DIM = len(next(iter(glove_embeddings.values())))
print('vacab size: ', VOCAB_SIZE ,'\n','embedding dim: ', EMBEDDED_DIM )

# Embeddomg matrix from tokenizer created before
embedding_matrix = np.zeros((VOCAB_SIZE,EMBEDDED_DIM))
for word, i in tokenizer.word_index.items():
    embedding_vector = glove_embeddings.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

In [ ]:
print(embedding_matrix.shape)
print(len(glove_embeddings))
glove_embeddings['happy']

In [ ]:
# after building embedding matrix, glove_embeddings dictionary can be deleted to save memory
del glove_embeddings, train_glove_oov, test_glove_oov
gc.collect()

In [ ]:
import numpy as np
from tensorflow.keras.callbacks import Callback
from tensorflow.keras.metrics import Precision, Recall # Precision, Recall metric이 모델 컴파일 시 포함되어 있어야 합니다.

class F1ScoreEarlyStopping(Callback):
    def __init__(self, patience=0, mode='auto', min_delta=0, verbose=0):
        super().__init__()
        self.patience = patience
        self.mode = mode
        self.min_delta = min_delta # 최소한 이만큼은 개선되어야 함
        self.verbose = verbose

        self.best_f1 = -np.inf if self.mode == 'max' else np.inf # F1이 높아질수록 좋으므로 초기값 -inf
        self.wait = 0 # 개선이 없는 에포크 수
        self.stopped_epoch = 0 # 학습이 중단된 에포크

        # mode 설정에 따른 비교 함수 결정
        if self.mode == 'min':
            self.monitor_op = np.less # 작아질수록 좋은 경우 (e.g., loss)
        elif self.mode == 'max':
            self.monitor_op = np.greater # 커질수록 좋은 경우 (e.g., accuracy, f1)
        else: # auto
            # F1-score는 높을수록 좋으므로 'max'로 가정
            self.monitor_op = np.greater


    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        # val_precision과 val_recall이 logs에 존재하는지 확인
        val_precision = logs.get('val_precision')
        val_recall = logs.get('val_recall')

        if val_precision is None or val_recall is None:
            if self.verbose > 0:
                print(f"Warning: val_precision or val_recall not found in logs. Cannot calculate F1-score for EarlyStopping.")
            return

        # F1-score 계산 (0으로 나누는 것을 방지)
        if (val_precision + val_recall) == 0:
            current_f1 = 0.0
        else:
            current_f1 = 2 * (val_precision * val_recall) / (val_precision + val_recall)
        
        # 로그에 현재 F1-score 추가 (옵션)
        logs['val_f1_score'] = current_f1
        current_lr_value = self.model.optimizer.learning_rate.numpy()
        logs['learning_rate'] = current_lr_value
        print(f" (Learning Rate: {current_lr_value:.8f})", end='')

        if self.monitor_op(current_f1, self.best_f1 + self.min_delta): # F1-score가 개선되었는지 확인
            self.best_f1 = current_f1
            self.wait = 0 # 개선되었으니 wait 카운트 리셋
            # Optional: save best weights if using restore_best_weights functionality
            # self.model.save_weights('best_model_f1.h5') 
        else:
            self.wait += 1 # 개선되지 않았으니 wait 카운트 증가
            if self.wait >= self.patience: # patience만큼 개선이 없으면
                self.stopped_epoch = epoch # 중단된 에포크 기록
                self.model.stop_training = True # 학습 중단 명령
                if self.verbose > 0:
                    print(f"\nEpoch {epoch+1}: Early stopping (val_f1_score did not improve for {self.patience} epochs).")
                    print(f"Best F1-score: {self.best_f1:.4f}")



# **Model building - LSTM using Meta features**

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, GRU, Dense, Concatenate, Flatten, Bidirectional, Dropout, RepeatVector
from tensorflow.keras.regularizers import l2

# hyper-papameters
MAX_LEN = max_length
LEARNING_RATE = 0.001
LAYER1_UNIT,LAYER2_UNIT =64,64
DROP_OUT1,DROP_OUTBI1,DROP_OUTBI2 =0.1,0.1,0.1
regularizer_lamda=0.01
NUM_EPOCHS = 150
BATCH_SIZE = 128
DECAY_STEPS = 10*dataset_size/BATCH_SIZE 
DECAY_RATE = 0.75

# 텍스트 시퀀스 입력 (padded_train의 shape와 일치)
text_input = Input(shape=(max_length,), name='text_input') # max_length는 이전 코드에서 계산된 최대 시퀀스 길이

# 임베딩 레이어
embedding_layer = Embedding(input_dim=VOCAB_SIZE, 
                            output_dim=EMBEDDED_DIM,
                            weights=[embedding_matrix],
                            input_length=max_length,
                            trainable=False)(text_input)





# 임베딩된 시퀀스를 처리하는 레이어 
lstm_layer1 = Bidirectional(LSTM(LAYER1_UNIT, return_sequences=True,dropout=DROP_OUTBI1,recurrent_dropout=DROP_OUTBI1))(embedding_layer)
lstm_layer2=Bidirectional(LSTM(LAYER2_UNIT,    dropout=DROP_OUTBI2,recurrent_dropout=DROP_OUTBI2))(lstm_layer1)
lstm_dropout=Dropout(DROP_OUT1)(lstm_layer2)

# 8. 결합된 특징을 처리하는 추가 레이어
x = Dense(32, activation='relu')(lstm_dropout)

# 9. 출력 레이어 (이진 분류)
output = Dense(1, activation='sigmoid')(x) 

# 10. 모델 정의: 입력과 출력을 지정
# 이제 모델은 텍스트 입력과 3가지 메타 특징 입력 (해시태그, 구두점, URL)을 받습니다.
model1 = Model(inputs=text_input, outputs=output)


lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=LEARNING_RATE,
    decay_steps=DECAY_STEPS,
    decay_rate=DECAY_RATE,
    staircase=True # 학습률이 계단식으로 감소 (False면 부드럽게 감소)
)

# Compile the model 
f1_early_stopping = F1ScoreEarlyStopping(patience=5, mode='max', verbose=1)

model1.compile(loss='binary_crossentropy',
              optimizer=Adam(learning_rate=lr_schedule, weight_decay=regularizer_lamda),
              metrics=['accuracy', tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.Precision(name='precision')])

model1.summary()


In [ ]:
# 모델 학습 (예시)
history_model1=model1.fit(
    padded_train, # 텍스트 입력과 해시태그 개수 입력을 리스트로 전달
    y_train,
    epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(padded_val, y_val),
    callbacks=f1_early_stopping
)

In [ ]:
# Plot results
model1_val_accuracy, model1_val_precision, model1_val_recall ,model1_val_f1_scores = plot_results(history_model1,model_name='Model 1', early_stopping_callback=f1_early_stopping)
print("model 1 accuracy", model1_val_accuracy)
print("model 1 precision", model1_val_precision)
print("model 1 recall",model1_val_recall)
print("model 1 f1 score",   model1_val_f1_scores )

y_pred_model1=plot_cfm(model1,X=padded_val, y_actual= y_val, model_name='Model1')

In [ ]:
# predict testing results
pred_prob = model1.predict(padded_test)
pred_prob = np.transpose(pred_prob)[0]
y_pred = list(map(lambda x: 0 if x < 0.5 else 1, pred_prob))

sample_submission["target"] = y_pred

sample_submission.to_csv("submission_lstm.csv", index=False)

# <a id="22"></a>

### Reference
https://www.kaggle.com/code/gunesevitan/nlp-with-disaster-tweets-eda-cleaning-and-bert <br>
https://www.kaggle.com/code/alexia/kerasnlp-starter-notebook-disaster-tweets <br>
https://www.kaggle.com/code/andreshg/nlp-glove-bert-tf-idf-lstm-explained <br>
https://www.kaggle.com/code/shahules/basic-eda-cleaning-and-glove<br>
https://www.kaggle.com/code/vbmokin/nlp-eda-bag-of-words-tf-idf-glove-bert
